# 02b — Follow-ups to Phase 2

Notebook 02 is left unchanged. Here we look more closely at two observations from it:

1. **ALAS2** (an erythroid gene) was the top-correlated gene for all six isotype controls and for
   15 proteins. Is this only an erythroid-vs-rest difference? We report signs, then repeat the
   all-gene ranking (a) without erythroid cells and (b) within cell types.
2. **Eight proteins with ~0 own-gene correlation**: one table with RNA detection, signal relative to
   isotype controls, own-gene r and the best gene.

No mapping is changed here. These are diagnostics.

In [1]:
import os, sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import h5py
import numpy as np
import pandas as pd
import scanpy as sc

from citeseq.paths import CITE_H5AD, REPO_ROOT, RESULTS_DIR
from citeseq.io import load_cite, _read_dataframe
from citeseq.labels import to_lineage
from citeseq.correlation import (load_protein_gene_map, mapped_log_expression,
                                 correlate_with_all_genes, within_group_gene_correlation)

MIN_CELLS = 50
SUBSET = int(os.environ.get("CITESEQ_SUBSET", 0)) or None
pd.set_option("display.float_format", "{:.3f}".format)

with h5py.File(CITE_H5AD) as f:
    var = _read_dataframe(f["var"])
mask = None
if SUBSET:
    mask = np.zeros(90_261, bool)
    mask[np.random.default_rng(0).choice(mask.size, SUBSET, replace=False)] = True
rna = load_cite(CITE_H5AD, "layers/counts", cell_mask=mask, var_mask=(var["feature_types"] == "GEX").to_numpy())
adt = load_cite(CITE_H5AD, "X", cell_mask=mask, var_mask=(var["feature_types"] == "ADT").to_numpy())

pmap = load_protein_gene_map(REPO_ROOT / "resources/protein_gene_map.csv", rna.var_names)
use = pmap[pmap["usable"]].reset_index(drop=True)
total = np.asarray(rna.X.sum(axis=1)).ravel()
R = mapped_log_expression(rna.X, rna.var_names, total, use["gene_list"].tolist())   # own-gene RNA
Pm = adt[:, use["protein"].tolist()].X.toarray()                                  # protein CLR
ISO = adt.obsm["ADT_isotype_controls"]
iso_names = list(ISO.columns)
ISO = ISO.to_numpy(dtype=np.float64)

sc.pp.normalize_total(rna, target_sum=1e4)   # log1p CP10k for all genes (per cell)
sc.pp.log1p(rna)
genes = np.asarray(rna.var_names)
cell_type = rna.obs["cell_type"].astype(str).to_numpy()
lineage = np.asarray(to_lineage(cell_type))
print(rna.shape, "| subset:", SUBSET)

(90261, 13953) | subset: None


## 1. ALAS2

Targets: the six isotype controls and the proteins whose top gene in notebook 02 was ALAS2.
Ranks are by signed r (rank 1 = highest positive r), as in notebook 02.

In [2]:
p2 = pd.read_csv(RESULTS_DIR / "phase2_protein_rna_correlation.csv")
alas2_proteins = p2.loc[p2["top_gene"] == "ALAS2", "protein"].tolist()
names = alas2_proteins + iso_names
cols = [use.index[use["protein"] == p][0] for p in alas2_proteins]
Y = np.hstack([Pm[:, cols], ISO])
ia = int(np.flatnonzero(genes == "ALAS2")[0])

def alas2_table(r, label):
    rows = []
    for k, name in enumerate(names):
        col = r[:, k]
        top_abs = int(np.nanargmax(np.abs(col)))
        rows.append({"target": name, "kind": "isotype" if name in iso_names else "protein",
                     f"ALAS2_r_{label}": col[ia],
                     f"ALAS2_rank_{label}": int(np.sum(col > col[ia]) + 1) if np.isfinite(col[ia]) else np.nan,
                     f"top_gene_{label}": genes[int(np.nanargmax(col))],
                     f"top_r_{label}": np.nanmax(col),
                     f"top_abs_gene_{label}": genes[top_abs],
                     f"top_abs_r_{label}": col[top_abs]})
    return pd.DataFrame(rows).set_index("target")

t0 = time.time()
r_all = correlate_with_all_genes(rna.X, Y)
ery = lineage == "Erythroid"
r_noery = correlate_with_all_genes(rna.X[~ery], Y[~ery])
r_within, n_types = within_group_gene_correlation(rna.X, Y, cell_type, MIN_CELLS)
print(f"done in {time.time() - t0:.0f}s; erythroid cells removed: {ery.sum():,}; cell types used: {n_types}")

alas = pd.concat([alas2_table(r_all, "all"), alas2_table(r_noery, "no_erythroid").iloc[:, 1:],
                  alas2_table(r_within, "within_type").iloc[:, 1:]], axis=1)
alas[["kind", "ALAS2_r_all", "ALAS2_rank_all", "top_abs_gene_all", "top_abs_r_all"]]

done in 10s; erythroid cells removed: 11,258; cell types used: 42


,kind,ALAS2_r_all,ALAS2_rank_all,top_abs_gene_all,top_abs_r_all
target,,,,,
CD154,protein,0.277,1,MT-ND3,-0.363
CD103,protein,0.237,1,ALAS2,0.237
CD152,protein,0.310,1,MT-ND3,-0.366
CD223,protein,0.191,1,MT-ND3,-0.289
CD134,protein,0.228,1,MT-ND3,-0.258
CD141,protein,0.289,1,RPS29,-0.345
CD58,protein,0.549,1,RPS27,-0.560
FceRIa,protein,0.496,1,ALAS2,0.496
CD137,protein,0.287,1,MT-ND3,-0.351


In [3]:
alas[["ALAS2_r_no_erythroid", "ALAS2_rank_no_erythroid", "top_gene_no_erythroid", "top_r_no_erythroid"]]

,ALAS2_r_no_erythroid,ALAS2_rank_no_erythroid,top_gene_no_erythroid,top_r_no_erythroid
target,,,,
CD154,-0.062,13448,RPLP0,0.135
CD103,-0.013,12772,ITGA1,0.143
CD152,-0.057,13417,RPS2,0.113
CD223,-0.043,13107,RPS2,0.098
CD134,-0.017,10540,RPL10,0.117
CD141,-0.052,13526,CST3,0.229
CD58,0.011,8915,MNDA,0.410
FceRIa,-0.013,12652,FCER1A,0.465
CD137,-0.047,13387,RPS2,0.103


In [4]:
alas[["ALAS2_r_within_type", "ALAS2_rank_within_type", "top_gene_within_type", "top_r_within_type"]]

,ALAS2_r_within_type,ALAS2_rank_within_type,top_gene_within_type,top_r_within_type
target,,,,
CD154,-0.053,13528,RPL10,0.085
CD103,-0.000,4837,RPLP0,0.036
CD152,-0.043,13192,RPS2,0.080
CD223,-0.038,12949,RPS2,0.080
CD134,-0.013,9820,RPS2,0.055
CD141,-0.032,13498,RPS26,0.069
CD58,0.042,1010,ACTB,0.150
FceRIa,0.013,267,FCER1A,0.095
CD137,-0.036,13075,RPS2,0.077


In [5]:
summary = alas.groupby("kind")[["ALAS2_r_all", "ALAS2_r_no_erythroid", "ALAS2_r_within_type",
                                 "ALAS2_rank_all", "ALAS2_rank_no_erythroid", "ALAS2_rank_within_type"]].median()
print("medians per target kind"); summary

medians per target kind


,ALAS2_r_all,ALAS2_r_no_erythroid,ALAS2_r_within_type,ALAS2_rank_all,ALAS2_rank_no_erythroid,ALAS2_rank_within_type
kind,,,,,,
isotype,0.144,-0.026,-0.021,1.000,13266.500,13143.000
protein,0.277,-0.044,-0.036,1.000,12993.000,13075.000


## 2. Eight proteins with ~0 own-gene correlation

* **RNA detected**: cells with >= 1 count of the mapped gene.
* **Signal vs. isotypes**: (i) median over cells of `protein CLR - mean isotype CLR`; (ii) share of
  cells where the protein CLR exceeds the highest isotype CLR. These are also computed in the cell type
  with the highest mean RNA of the mapped gene (types with >= 50 cells).
* Caveat: isotype antibodies differ in concentration and affinity from the specific antibodies, so this
  comparison is rough.

In [6]:
low8 = ["CD83", "CD115", "CD103", "CD163", "CD154", "CD223", "CD152", "CD42b"]
iso_mean, iso_max = ISO.mean(axis=1), ISO.max(axis=1)
big = pd.Series(cell_type).value_counts()
big = big.index[big >= MIN_CELLS]
rows = []
for p in low8:
    j = int(use.index[use["protein"] == p][0])
    x, y = R[:, j], Pm[:, j]
    type_means = pd.Series(x).groupby(cell_type).mean().loc[big]
    top_t = type_means.idxmax()
    sel = cell_type == top_t
    row2 = p2.set_index("protein").loc[p]
    rows.append({
        "protein": p, "gene": use.loc[j, "genes"],
        "rna_detected_cells": int((x > 0).sum()), "rna_detected_frac": (x > 0).mean(),
        "median_clr_minus_iso": np.median(y - iso_mean), "frac_above_max_iso": (y > iso_max).mean(),
        "top_rna_type": top_t, "top_type_n": int(sel.sum()),
        "top_type_rna_detected_frac": (x[sel] > 0).mean(),
        "top_type_median_clr_minus_iso": np.median(y[sel] - iso_mean[sel]),
        "top_type_frac_above_max_iso": (y[sel] > iso_max[sel]).mean(),
        "own_gene_r": row2["pearson_all"], "top_gene": row2["top_gene"], "top_gene_r": row2["top_gene_r"],
    })
low = pd.DataFrame(rows).set_index("protein")
low

,gene,rna_detected_cells,rna_detected_frac,median_clr_minus_iso,frac_above_max_iso,top_rna_type,top_type_n,top_type_rna_detected_frac,top_type_median_clr_minus_iso,top_type_frac_above_max_iso,own_gene_r,top_gene,top_gene_r
protein,,,,,,,,,,,,,
CD83,CD83,8099,0.090,1.134,0.984,Naive CD20+ B IGKC-,1979,0.321,1.148,0.984,-0.065,ALAS2,0.248
CD115,CSF1R,8727,0.097,0.338,0.544,CD16+ Mono,2635,0.717,0.340,0.576,-0.090,ALAS2,0.615
CD103,ITGAE,15924,0.176,-0.048,0.077,Transitional B,1575,0.383,-0.084,0.036,-0.021,ALAS2,0.237
CD163,CD163,6915,0.077,0.470,0.685,CD14+ Mono,21693,0.288,0.473,0.704,-0.016,ALAS2,0.177
CD154,CD40LG,4192,0.046,1.004,0.961,CD4+ T CD314+ CD45RA+,93,0.355,0.751,1.000,-0.025,ALAS2,0.277
CD223,LAG3,1979,0.022,0.324,0.521,gdT TCRVD2+,191,0.220,0.308,0.518,-0.021,ALAS2,0.191
CD152,CTLA4,1056,0.012,0.640,0.832,T reg,609,0.305,0.643,0.836,-0.017,ALAS2,0.310
CD42b,GP1BA,325,0.004,-0.076,0.056,MK/E prog,690,0.055,-0.104,0.038,0.001,FCN1,0.159


In [7]:
# Reference: the same signal-vs-isotype numbers for all usable proteins, to see where the eight fall
ref = pd.DataFrame({"protein": use["protein"],
                    "median_clr_minus_iso": np.median(Pm - iso_mean[:, None], axis=0),
                    "frac_above_max_iso": (Pm > iso_max[:, None]).mean(axis=0)})
ref["in_low8"] = ref["protein"].isin(low8)
print(ref.groupby("in_low8")[["median_clr_minus_iso", "frac_above_max_iso"]].median())

         median_clr_minus_iso  frac_above_max_iso
in_low8                                          
False                   0.303               0.490
True                    0.404               0.615


In [8]:
# CD115 in detail
j = int(use.index[use["protein"] == "CD115"][0])
mono = cell_type == "CD14+ Mono"
print("CD115 top gene (notebook 02):", p2.set_index("protein").loc["CD115", ["top_gene", "top_gene_r"]].to_dict())
print(f"CSF1R detected in {int((R[:, j] > 0).sum()):,} of {len(R):,} cells ({(R[:, j] > 0).mean():.1%})")
print(f"CSF1R detected in CD14+ Mono: {int((R[mono, j] > 0).sum()):,} of {int(mono.sum()):,} ({(R[mono, j] > 0).mean():.1%})")
print("median CD115 CLR, CD14+ Mono vs erythroid:", round(np.median(Pm[mono, j]), 3), "vs", round(np.median(Pm[ery, j]), 3))

CD115 top gene (notebook 02): {'top_gene': 'ALAS2', 'top_gene_r': 0.6147079430516201}
CSF1R detected in 8,727 of 90,261 cells (9.7%)
CSF1R detected in CD14+ Mono: 5,414 of 21,693 (25.0%)
median CD115 CLR, CD14+ Mono vs erythroid: 0.484 vs 1.273


In [9]:
if SUBSET is None:
    alas.to_csv(RESULTS_DIR / "phase2b_alas2.csv")
    low.to_csv(RESULTS_DIR / "phase2b_low_correlation_proteins.csv")
import resource, platform
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"peak memory: {(peak / 1e9 if platform.system() == 'Darwin' else peak * 1024 / 1e9):.2f} GB")

peak memory: 3.34 GB


## Findings (full data)

**ALAS2 is an erythroid-vs-rest axis, not a cell-level relationship.**
* Over all cells ALAS2 correlates **positively** with all six isotypes (r 0.11-0.20) and with the 15
  proteins (0.18-0.67). By |r| the strongest gene is often **negatively** correlated (MT-ND3,
  TMSB4X, RPS29, B2M; r down to -0.56). Both point to the same split: these CLR values are higher in
  erythroid cells.
* **Without erythroid cells** (11,258 removed) ALAS2's r drops to about 0 (median -0.03 isotypes,
  -0.04 proteins). Its rank falls from 1 to ~13,000 of 13,953.
* **Within cell types** (42 types) the picture is the same: ALAS2 r about 0, rank ~13,000. The top genes
  become ribosomal genes (RPS2, RPL10, RPLP0) with small r (<= 0.15).
* Without erythroid cells, a few proteins show an expected partner gene: FceRIa -> FCER1A (its own
  gene, r 0.47), CD88 -> AIF1, CD58 -> MNDA, CD141 -> CST3.
* **Hypothesis, not tested:** the lower isotype share in erythroid cells (notebook 02) may relate to Fc
  receptors. Myeloid cells carry Fc receptors that can bind antibodies non-specifically; erythroid
  cells largely do not.

**Eight low-correlation proteins.**
* CD103 and CD42b: protein CLR is at isotype level (median difference about -0.05 to -0.08; only 4-8%
  of cells above the highest isotype), also in the cell type with the most RNA. This fits a
  background-level signal.
* The other six (CD83, CD115, CD163, CD154, CD223, CD152) are **above** isotype level in most cells
  (52-98%), yet do not follow their own gene's RNA. On this crude measure the eight are not lower than
  the other proteins (median CLR - isotype 0.40 vs 0.30). So the notebook-02 hypothesis
  "signal close to background" holds only for CD103 and CD42b.
* RNA detection of the mapped genes is low: 0.4% (GP1BA) to 17.6% (ITGAE) of cells.
* **CD115:** top gene ALAS2 (r 0.61). CSF1R is detected in 8,727 cells (9.7%), in 25% of CD14+ Mono
  and 72% of CD16+ Mono. The CD115 protein signal is higher in erythroid cells (median CLR 1.27) than
  in CD14+ Mono (0.48), i.e. highest where the gene is not expressed. A non-specific erythroid signal is
  a hypothesis, not tested.

Peak memory 3.3 GB, runtime < 1 min.